In [ ]:
# 1. Montar Google Drive
# Aqui vive todo: el pipeline (notebooks + libreria comun), los datasets
# bajados de Roboflow, los runs de entrenamiento y los .pt finales.
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Dependencias del entrenamiento (YOLO + cliente de Roboflow)
!pip install ultralytics roboflow

In [ ]:
# 3. Pararse en la carpeta train/ del proyecto en Drive
# Todo lo que se genere despues (datasets, runs, models) queda dentro de ella.
%cd /content/drive/MyDrive/CNM-PROJECT/CNM-Brazo_Robotico_Cajas/train

In [ ]:
# 4. Directorio base + libreria compartida
# pipeline/common/yolo_train_utils.py tiene la logica repetida entre notebooks
# (descarga de Roboflow, validacion de data.yaml, busqueda de best.pt, guardado
# de pesos). Se agrega pipeline/ al sys.path para poder importarla.
import os
import sys

BASE_PATH = os.getcwd()
os.makedirs(BASE_PATH, exist_ok=True)

PIPELINE_PATH = os.path.join(BASE_PATH, "pipeline")
if PIPELINE_PATH not in sys.path:
    sys.path.append(PIPELINE_PATH)

print(f"Directorio de trabajo detectado: {BASE_PATH}")

In [ ]:
# 5. Imports del entrenamiento
import logging
from ultralytics import YOLO
from common.yolo_train_utils import (
    setup_logging,
    download_roboflow_dataset,
    validate_data_yaml,
    find_best_checkpoint,
    save_model_with_backup,
)

setup_logging()

In [ ]:
# 6. Descargar el dataset de brazo robotico desde Roboflow
# Cambia workspace/project_name/version_num aqui si se etiqueta una nueva
# version del dataset; el resto del notebook no necesita tocarse.
# La key NUNCA va hardcodeada aca -- setear ROBOFLOW_API_KEY antes de correr
# esta celda (en Colab: os.environ["ROBOFLOW_API_KEY"] = getpass.getpass()).
dataset = download_roboflow_dataset(
    api_key=os.environ["ROBOFLOW_API_KEY"],
    workspace="alys-peru",
    project_name="cnm_palletsbrazo",
    version_num=1,
)

In [ ]:
# 7. Validar que el dataset trajo su data.yaml (clases + rutas de train/val)
# antes de gastar tiempo de GPU entrenando con un dataset roto o incompleto.
data_yaml = validate_data_yaml(dataset.location)

In [ ]:
# 8. Cargar el modelo base preentrenado de Ultralytics
logging.info("Cargando modelo base...")
model = YOLO("yolo26s.pt")

In [ ]:
# 9. Entrenamiento
# RUN_NAME identifica esta corrida: nombra la carpeta en runs/detect/<RUN_NAME>*
# y luego la carpeta final en models/<RUN_NAME>/. Si se corre de nuevo,
# Ultralytics le agrega un sufijo numerico (RUN_NAME2, RUN_NAME3...) sin pisar
# el run anterior.
import torch

DEVICE = 0 if torch.cuda.is_available() else "cpu"
if DEVICE == "cpu":
    logging.warning("No se detecto GPU, entrenando en CPU (va a ser MUY lento). Revisa Entorno de ejecucion > GPU en Colab.")

RUN_NAME = "robotic-arm"

logging.info("Iniciando entrenamiento...")

model.train(
    data=data_yaml,
    epochs=150,
    patience=40,
    device=DEVICE,
    batch=96,
    imgsz=512,          # alineado al resize real del dataset (Roboflow: stretch 512x512)
    name=RUN_NAME,

    # --- Augmentations de color/luz (cubren variacion de brillo e iluminacion) ---
    hsv_h=0.015,        # variacion leve de tono (default)
    hsv_s=0.5,          # saturacion, ya viene algo cubierto por Roboflow pero refuerza
    hsv_v=0.6,          # brillo/exposicion — sube el rango para simular mas condiciones de luz

    # --- Augmentations geometricas ---
    degrees=5.0,        # rotacion leve
    translate=0.1,
    scale=0.3,          # zoom in/out, complementa el crop 0-25% de Roboflow
    shear=0.0,
    perspective=0.0,
    flipud=0.0,
    fliplr=0.5,

    # --- Mezcla ---
    mosaic=1.0,
    mixup=0.1,
)

logging.info("Entrenamiento finalizado")

In [ ]:
# 10. Ubicar el best.pt de la corrida que acaba de terminar
# (busca en runs/detect/ la carpeta RUN_NAME* mas reciente).
best_pt_path = find_best_checkpoint(BASE_PATH, RUN_NAME)

In [ ]:
# 11. Publicar el modelo entrenado en models/<RUN_NAME>/
# - Si ya existia un .pt de una corrida anterior, lo respalda con timestamp.
# - Copia el best.pt nuevo como <RUN_NAME>.pt (el nombre "vigente" siempre es ese).
# - Escribe classes.txt junto al peso, con el indice y nombre de cada clase
#   que aprendio ese modelo especifico (util para saber que .pt es cual sin
#   tener que cargarlo).
model_dir = os.path.join(BASE_PATH, "models", RUN_NAME)
dest_path = save_model_with_backup(best_pt_path, model_dir, RUN_NAME, model.names)

logging.info("Proceso completado correctamente")

In [ ]:
# 12. Desmontar Drive para asegurar que todo quedo escrito a disco
drive.flush_and_unmount()
print("Drive desmontado")